In [6]:
# -*- coding: utf-8 -*-
# CT -> MRI(b=0) Latent Diffusion (DDPM + DDIM)
# - 잠재공간 z에서 학습/샘플링 (VAE encode/decode)
# - 조건: CT를 잠재 해상도로 다운샘플 + 1x1 conv로 채널 맞춰 concat
# - 손실: latent ε-MSE (+ 선택: x̂0 복원 후 TV/LPIPS 보조)
# - 에폭별: CT-only inference(EMA 가중치) + Triptych(Input/GT/Pred)


In [1]:
import sys
import os

# ldm 폴더를 PYTHONPATH에 추가
ldm_path = os.path.expanduser('~/CT2MRI/diffusion_model/ldm')
if ldm_path not in sys.path:
    sys.path.insert(0, ldm_path)

print(f"Added to path: {ldm_path}")

# 이제 import 가능
from taming.modules.vqvae.quantize import VectorQuantizer2 as VectorQuantizer

Added to path: /home/youkyoung/CT2MRI/diffusion_model/ldm


In [2]:
import os, glob, time, random, warnings, math
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# ===== Repo imports =====
import sys
sys.path.append('.')  # yk_test.ipynb와 ldm/가 같은 루트라고 가정

from ldm.modules.diffusionmodules.openaimodel import UNetModel
from ldm.modules.diffusionmodules.util import make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters

# VAE (사용할 클래스명을 여기에서 import — 아래 build_vae()에서 다양한 시그니처를 흡수)
from ldm.models.autoencoder import AutoencoderKL  # 필요시 다른 이름으로 교체
from ldm.models.autoencoder import VQModelInterface 

# (선택) LPIPS가 있다면 사용 가능
try:
    from ldm.modules.losses.lpips import LPIPS
    HAS_LPIPS = True
except Exception:
    HAS_LPIPS = False

/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

# =========================
# Config
# =========================
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # ===== 데이터 루트 (학습용) =====
    # "data_root": "/data/ct-mri/dataset/processed_final_5",
    "data_root": "../data/processed_with_skullstrip/",
    "use_groups": ["bleed", "non_bleed"],
    "ct_dir_candidates": ["ct_axial", "ct", "ct_png", "ctjpg", "ctjpeg"],
    "mri_b0_dir_candidates": ["mri_b0_axial", "mri_b0", "b0", "dwi_b0", "dwi", "dwi_axial_b0"],
    "ct_file_globs": ["*ct*.png", "*CT*.png"],
    "mri_b0_file_globs": ["*b0*.png", "*B0*.png", "*dwi*.png", "*DWI*.png"],

    # ===== 학습 하이퍼 =====
    "img_size": 256,           # 입력 픽셀 해상도 (VAE가 내부적으로 다운)
    "batch_size": 8,
    "num_workers": 4,
    "epochs": 50,
    "lr": 2e-4,
    "ema_decay": 0.999,

    # ===== 확산 스케줄 =====
    "num_timesteps": 1000,
    "beta_schedule": "cosine",
    "ddim_steps": 200,         # latent에서는 steps↑ 추천
    "ddim_eta": 0.0,

    # ===== VAE =====
    # scale은 AutoencoderKL(LDM 계열) 보통 0.18215. (VAE에 따라 다를 수 있음)
    "vae": {
        "scale": 0.18215,
        "ckpt": None,          # 필요시 외부 ckpt 경로; 없으면 AutoencoderKL.from_config 등 사용
        "eval_mode": True,
        "freeze": True
    },

    # ===== Latent U-Net =====
    # z_channels는 VAE의 latent 채널 수 (SD 계열=4가 일반적)
    # cond_channels는 CT 임베딩 채널 수 (1~4 추천)
    "latent_unet": {
        "z_channels": 4,
        "cond_channels": 1,
        "model_channels": 192,
        "num_res_blocks": 2,
        "channel_mult": (1,2,3,4),
        "attention_resolutions": (16,),  # latent 해상도 기준이 아니라 입력 image_size 기준: 여기선 무시되어도 OK
        "num_heads": -1,
        "num_head_channels": 64,
        "num_heads_upsample": -1,
        "dropout": 0.1,
        "use_checkpoint": False,
        "use_fp16": False,
        "use_scale_shift_norm": True,
        "resblock_updown": False,
        "use_new_attention_order": False,
        "use_spatial_transformer": False,  # cross-attn은 다음 단계에서
        "transformer_depth": 1,
        "context_dim": None,
        "n_embed": None,
        "legacy": True,
    },

    # ===== 보조 손실 옵션 =====
    "aux_losses": {
        "use_tv": True,
        "tv_weight": 1e-5,
        "use_lpips": False,              # LPIPS 다운 받았으면 True로
        "lpips_weight": 0.05
    },

    # ===== 체크포인트 =====
    "save_dir": "./ckpt_ct2mri_b0_ldm_strip",
    "resume": None,   # "./ckpt_ct2mri_b0_ldm/last.pt"

    # ===== 에폭별 CT-only inference =====
    "inference": {
        "enabled": True,
        "num_examples": 8,
        "sampler": "ddim",     # "ddim" or "ddpm"
        "ddim_steps": 200,
        "eta": 0.0,
        "seed": 1234,
        "ct_list": [],
        "ct_glob": None,
        "out_subdir": "epoch_infer_ldm",
        "write_mapping": True
    }
}


In [4]:
# =========================
# Utils
# =========================
def set_seed(s: int):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def imread_gray(path: str, size: int) -> np.ndarray:
    img = Image.open(path).convert("L").resize((size, size), Image.BICUBIC)
    return np.asarray(img, dtype=np.float32) / 255.0

def to_tensor01(img01: np.ndarray) -> torch.Tensor:
    x = torch.from_numpy(img01)[None, ...]   # (1,H,W) [0,1]
    return x

def to_range_m11(x01: torch.Tensor) -> torch.Tensor:
    return x01 * 2.0 - 1.0

def to_image_from_m11(x: torch.Tensor) -> Image.Image:
    x = x.detach().clamp(-1,1)
    x = (x + 1.0) * 0.5
    return Image.fromarray((x[0].cpu().numpy()*255).round().astype(np.uint8), "L")

def save_triptych(ct_t_m11: torch.Tensor, pred_t_m11: torch.Tensor, gt_path: Optional[str],
                  out_path: str, size: int):
    ct_p   = to_image_from_m11(ct_t_m11)
    pred_p = to_image_from_m11(pred_t_m11)
    if gt_path and os.path.exists(gt_path):
        gt_arr = imread_gray(gt_path, size)
        gt_p   = Image.fromarray((gt_arr*255).astype(np.uint8), "L")
    else:
        gt_p   = Image.new("L", (size,size), 0)
    panel = Image.new("L", (size*3, size))
    panel.paste(ct_p,(0,0)); panel.paste(gt_p,(size,0)); panel.paste(pred_p,(size*2,0))
    os.makedirs(os.path.dirname(out_path), exist_ok=True); panel.save(out_path)

def _find_first_existing_dir(base: str, candidates_lower: List[str]) -> str:
    entries = [d for d in glob.glob(os.path.join(base, "*")) if os.path.isdir(d)]
    name2path = {os.path.basename(p).lower(): p for p in entries}
    for cand in candidates_lower:
        if cand in name2path: return name2path[cand]
    return ""

def _glob_files(base: str, patterns: List[str]) -> List[str]:
    out = []
    for pat in patterns: out.extend(glob.glob(os.path.join(base, pat)))
    return sorted(set(out))

def _collect_modality_files(patient_dir: str, dir_candidates: List[str], file_globs: List[str]) -> List[str]:
    found_dir = _find_first_existing_dir(patient_dir, [c.lower() for c in dir_candidates])
    if found_dir:
        # files = sorted(glob.glob(os.path.join(found_dir, "*.jpg")))
        files = sorted(glob.glob(os.path.join(found_dir, "*.png")))
        if files: return files
    return _glob_files(patient_dir, file_globs)

def _pair_by_filename(ct_files: List[str], mri_files: List[str]) -> List[Tuple[str,str]]:
    m = {os.path.splitext(os.path.basename(p))[0]: p for p in mri_files}
    pairs = []
    for c in ct_files:
        stem = os.path.splitext(os.path.basename(c))[0]
        if stem in m: pairs.append((c, m[stem]))
    return pairs

def discover_pairs(data_root: str, use_groups: List[str],
                   ct_dir_candidates: List[str], mri_b0_dir_candidates: List[str],
                   ct_file_globs: List[str], mri_b0_file_globs: List[str]) -> List[Tuple[str,str]]:
    all_pairs = []
    for g in use_groups:
        group_dir = os.path.join(data_root, g)
        if not os.path.isdir(group_dir): continue
        patients = sorted([p for p in glob.glob(os.path.join(group_dir, "*")) if os.path.isdir(p)])
        for pd in patients:
            ct_files  = _collect_modality_files(pd, ct_dir_candidates,  ct_file_globs)
            mri_files = _collect_modality_files(pd, mri_b0_dir_candidates, mri_b0_file_globs)
            if not ct_files or not mri_files: continue
            pairs = _pair_by_filename(ct_files, mri_files)      # 🔒 파일명 일치만
            if not pairs:
                warnings.warn(f"[Skip patient - no filename match] {pd}")
                continue
            all_pairs.extend(pairs)
    if not all_pairs: raise RuntimeError(f"No CT-MRI(b0) pairs under {data_root}")
    return all_pairs



In [5]:
# =========================
# Dataset
# =========================
class CTMRIFolderDataset(Dataset):
    def __init__(self, cfg: Dict):
        self.cfg = cfg
        self.pairs = discover_pairs(
            cfg["data_root"], cfg["use_groups"],
            cfg["ct_dir_candidates"], cfg["mri_b0_dir_candidates"],
            cfg["ct_file_globs"], cfg["mri_b0_file_globs"]
        )
        self.size = cfg["img_size"]
        print(f"[Dataset] discovered pairs: {len(self.pairs)}")

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        ct_path, mri_path = self.pairs[idx]
        ct01  = to_tensor01(imread_gray(ct_path,  self.size))   # (1,H,W) [0,1]
        mri01 = to_tensor01(imread_gray(mri_path, self.size))   # (1,H,W) [0,1]
        return {
            "ct_m11":  to_range_m11(ct01),     # [-1,1]
            "mri_m11": to_range_m11(mri01),    # [-1,1]
            "ct_path": ct_path,
            "mri_path": mri_path
        }


In [6]:
from ldm.util import instantiate_from_config
# KL이 어디에 정의됐는지 레포별로 다를 수 있어 안전하게 import
try:
    from ldm.models.autoencoder import AutoencoderKL, VQModelInterface
except ImportError:
    try:
        from ldm.models.autoencoder_kl import AutoencoderKL
        from ldm.models.autoencoder import VQModelInterface
    except Exception:
        AutoencoderKL = None
        VQModelInterface = None


class VAEWrapper(nn.Module):
    """
    - encode(x): 1ch 입력이면 3ch로 repeat → KL의 posterior.sample() 후 0.18215 스케일 적용
    - decode(z): 스케일 되돌린 뒤 디코드 → 3ch면 평균내어 1ch로 반환
    """
    def __init__(self, encode_fn, decode_fn):
        super().__init__()
        self._enc = encode_fn
        self._dec = decode_fn

    @torch.no_grad()
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self._enc(x)

    @torch.no_grad()
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self._dec(z)


def build_vae(cfg):
    vae_cfg = cfg["vae"]
    from omegaconf import OmegaConf
    raw = OmegaConf.load(vae_cfg["config"])
    conf = OmegaConf.to_container(raw, resolve=True)
    ckpt_path = vae_cfg.get("ckpt", None)
    assert ckpt_path and os.path.exists(ckpt_path), f"VAE ckpt 경로가 필요합니다: {ckpt_path}"

    # 1) 최상위에 KL이 오는 형태인지 확인
    top = conf.get("model", {})
    top_target = top.get("target", "")
    if "AutoencoderKL" in top_target:
        # ---- KL at top-level ----
        assert AutoencoderKL is not None, "AutoencoderKL 클래스를 찾을 수 없습니다."
        kl = instantiate_from_config(top)   # KL 인스턴스화
        kl.to(cfg["device"])                      # ✅ 추가: 모델을 GPU/CPU로 이동
        sd_all = torch.load(ckpt_path, map_location="cpu")
        state = sd_all.get("state_dict", sd_all)
        # 일부 ckpt는 first_stage_model.* 없이 KL만 들어있음 → 그대로 시도
        missing, unexpected = kl.load_state_dict(state, strict=False)
        if vae_cfg.get("eval_mode", True): kl.eval()
        if vae_cfg.get("freeze", True):
            for p in kl.parameters(): p.requires_grad_(False)

        # 스케일 관례(0.18215): KL 계열은 z에 scale 곱/나눔을 적용하는 게 일반적
        scale = 0.18215
        # 채널 적응: KL은 보통 RGB(3ch)로 학습 → 1ch 입력은 repeat, 3ch 출력은 평균
        def _enc(x_m11):
            x = x_m11.repeat(1,3,1,1) if x_m11.size(1)==1 else x_m11
            posterior = kl.encode(x)
            z = posterior.sample() * scale
            return z
        def _dec(z):
            x = kl.decode(z / scale)
            return x.mean(1, keepdim=True) if x.size(1)==3 else x
        # z_channels 동기화
        zc = top["params"]["ddconfig"]["z_channels"]
        if cfg["latent_unet"]["z_channels"] != zc:
            print(f"[VAE/KL] z_channels {cfg['latent_unet']['z_channels']} -> {zc}")
            cfg["latent_unet"]["z_channels"] = zc
        return VAEWrapper(_enc, _dec)

    # 2) 아니라면 first_stage_config를 찾아서 (VQ 또는 KL) 처리
    def find_first_stage_config(d: dict):
        for path in [["model","params","first_stage_config"],
                     ["model","first_stage_config"],["first_stage_config"]]:
            cur=d; ok=True
            for k in path:
                if isinstance(cur, dict) and k in cur: cur=cur[k]
                else: ok=False; break
            if ok and "target" in cur: return cur
        return None

    fs = find_first_stage_config(conf)
    assert fs and "target" in fs, "config에서 VAE 구성을 찾지 못했습니다."
    target = fs["target"]

    sd_all = torch.load(ckpt_path, map_location="cpu")
    state = sd_all.get("state_dict", sd_all)
    prefix = "first_stage_model."
    vae_sd = {k[len(prefix):]: v for k,v in state.items() if k.startswith(prefix)} or state

    if "AutoencoderKL" in target:
        assert AutoencoderKL is not None, "AutoencoderKL 클래스를 찾을 수 없습니다."
        kl = instantiate_from_config(fs)
        kl.to(cfg["device"])                      # ✅ 추가
        missing, unexpected = kl.load_state_dict(vae_sd, strict=False)
        if vae_cfg.get("eval_mode", True): kl.eval()
        if vae_cfg.get("freeze", True):
            for p in kl.parameters(): p.requires_grad_(False)
        scale = 0.18215
        def _enc(x_m11):
            x = x_m11.repeat(1,3,1,1) if x_m11.size(1)==1 else x_m11
            posterior = kl.encode(x); z = posterior.sample() * scale
            return z
        def _dec(z):
            x = kl.decode(z / scale)
            return x.mean(1, keepdim=True) if x.size(1)==3 else x
        zc = fs["params"]["ddconfig"]["z_channels"]
        if cfg["latent_unet"]["z_channels"] != zc:
            print(f"[VAE/KL] z_channels {cfg['latent_unet']['z_channels']} -> {zc}")
            cfg["latent_unet"]["z_channels"] = zc
        return VAEWrapper(_enc, _dec)

    else:  # VQModelInterface
        assert VQModelInterface is not None, "VQModelInterface 클래스를 찾을 수 없습니다."
        vq = instantiate_from_config(fs)
        vq.to(cfg["device"])                      # ✅ 추가
        cur = vq.state_dict()
        filt = {k:v for k,v in vae_sd.items() if k in cur and cur[k].shape==v.shape}
        vq.load_state_dict(filt, strict=False)
        if vae_cfg.get("eval_mode", True): vq.eval()
        if vae_cfg.get("freeze", True):
            for p in vq.parameters(): p.requires_grad_(False)
        def _enc(x_m11):
            x = x_m11.repeat(1,3,1,1) if x_m11.size(1)==1 else x_m11
            return vq.encode(x)
        def _dec(z):
            x = vq.decode(z, force_not_quantize=True)
            return x.mean(1, keepdim=True) if x.size(1)==3 else x
        zc = fs["params"].get("embed_dim", fs["params"]["ddconfig"]["z_channels"])
        if cfg["latent_unet"]["z_channels"] != zc:
            print(f"[VAE/VQ] z_channels {cfg['latent_unet']['z_channels']} -> {zc}")
            cfg["latent_unet"]["z_channels"] = zc
        return VAEWrapper(_enc, _dec)

In [7]:
# =========================
# Diffusion schedule
# =========================
@dataclass
class DiffusionSchedule:
    betas: torch.Tensor
    alphas: torch.Tensor
    alphas_cumprod: torch.Tensor
    alphas_cumprod_prev: torch.Tensor
    sqrt_alphas_cumprod: torch.Tensor
    sqrt_one_minus_alphas_cumprod: torch.Tensor
    posterior_variance: torch.Tensor

def build_schedule(num_steps: int, beta_schedule="cosine", device="cpu") -> DiffusionSchedule:
    betas = torch.from_numpy(make_beta_schedule(beta_schedule, num_steps)).float().to(device)
    alphas = 1.0 - betas
    a_bar = torch.cumprod(alphas, dim=0)
    a_bar_prev = torch.cat([torch.tensor([1.0], device=device), a_bar[:-1]], dim=0)
    return DiffusionSchedule(
        betas=betas, alphas=alphas,
        alphas_cumprod=a_bar, alphas_cumprod_prev=a_bar_prev,
        sqrt_alphas_cumprod=torch.sqrt(a_bar),
        sqrt_one_minus_alphas_cumprod=torch.sqrt(1.0 - a_bar),
        posterior_variance=betas * (1.0 - a_bar_prev) / (1.0 - a_bar)
    )


In [8]:
# =========================
# Latent U-Net + Cond encoder
# =========================
class CTCondEncoder(nn.Module):
    """ CT(1ch, H,W) -> (cond_channels, H/4, W/4) """
    def __init__(self, cond_channels=1):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(1, 32, 3, 2, 1), nn.SiLU(),      # H/2
            nn.Conv2d(32, 64, 3, 2, 1), nn.SiLU(),     # H/4
            nn.Conv2d(64, cond_channels, 1)            # channel match
        )
    def forward(self, x):  # x in [-1,1]
        return self.enc(x)

def build_latent_unet(cfg, latent_shape_hw: int):
    lu = cfg["latent_unet"]
    in_ch  = lu["z_channels"] + lu["cond_channels"]
    out_ch = lu["z_channels"]
    # image_size 인자는 일부 구현에서만 쓰임. 안전하게 넣어둠(잠재 해상도 추정)
    net = UNetModel(
        image_size=latent_shape_hw,  # 예: 64 (256/4)
        in_channels=in_ch,
        out_channels=out_ch,
        model_channels=lu["model_channels"],
        num_res_blocks=lu["num_res_blocks"],
        channel_mult=lu["channel_mult"],
        attention_resolutions=lu["attention_resolutions"],
        num_heads=lu["num_heads"],
        num_head_channels=lu["num_head_channels"],
        num_heads_upsample=lu["num_heads_upsample"],
        dropout=lu["dropout"],
        conv_resample=True, dims=2, num_classes=None,
        use_checkpoint=lu["use_checkpoint"],
        use_fp16=lu["use_fp16"],
        use_scale_shift_norm=lu["use_scale_shift_norm"],
        resblock_updown=lu["resblock_updown"],
        use_new_attention_order=lu["use_new_attention_order"],
        use_spatial_transformer=lu["use_spatial_transformer"],
        transformer_depth=lu["transformer_depth"],
        context_dim=lu["context_dim"],
        n_embed=lu["n_embed"],
        legacy=lu["legacy"]
    )
    cond_enc = CTCondEncoder(cond_channels=lu["cond_channels"])
    return net, cond_enc

In [9]:

# =========================
# Training / Sampling (latent)
# =========================
def q_sample_latent(z0, t, sched: DiffusionSchedule, noise=None):
    if noise is None: noise = torch.randn_like(z0)
    s_ab = sched.sqrt_alphas_cumprod[t].view(-1,1,1,1)
    s_om = sched.sqrt_one_minus_alphas_cumprod[t].view(-1,1,1,1)
    return s_ab * z0 + s_om * noise, noise

@torch.no_grad()
def sample_ddpm_latent(model, cond_enc, vae_wrap, ct_m11, H, sched: DiffusionSchedule, device="cpu"):
    # ct_m11: (B,1,H,W) [-1,1] → cond: (B,Cc,H/4,W/4)
    B = ct_m11.size(0)
    ct_cond = cond_enc(ct_m11)                         # (B,Cc,h,w)
    Cz = CFG["latent_unet"]["z_channels"]
    lat_h = ct_cond.shape[-2]; lat_w = ct_cond.shape[-1]
    z = torch.randn(B, Cz, lat_h, lat_w, device=device)
    T = sched.betas.size(0)
    for t in reversed(range(T)):
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([z, ct_cond], 1).float(), t_b)
        a_bar = sched.alphas_cumprod[t]
        z0 = (z - torch.sqrt(1 - a_bar) * eps) / torch.sqrt(a_bar)
        mean = torch.sqrt(sched.alphas_cumprod_prev[t]) * z0 + torch.sqrt(1.0 - sched.alphas_cumprod_prev[t]) * eps
        z = mean + (torch.sqrt(sched.posterior_variance[t]) * torch.randn_like(z) if t>0 else 0)
    x_pred_m11 = vae_wrap.decode(z)                    # (B,1,H,W) in [-1,1]
    return x_pred_m11

@torch.no_grad()
def sample_ddim_latent(model, cond_enc, vae_wrap, ct_m11, H, sched: DiffusionSchedule,
                       ddim_steps=200, eta=0.0, device="cpu"):
    B = ct_m11.size(0)
    ct_cond = cond_enc(ct_m11)
    Cz = CFG["latent_unet"]["z_channels"]
    lat_h = ct_cond.shape[-2]; lat_w = ct_cond.shape[-1]
    z = torch.randn(B, Cz, lat_h, lat_w, device=device)

    ts = np.array(make_ddim_timesteps('uniform', ddim_steps, sched.betas.size(0))) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(sched.alphas_cumprod.cpu().numpy(), ts, eta)
    sigmas = torch.from_numpy(sigmas).float().to(device)
    alphas = torch.from_numpy(alphas).float().to(device)
    alphas_prev = torch.from_numpy(alphas_prev).float().to(device)

    for i in reversed(range(len(ts))):
        t = int(ts[i]); t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([z, ct_cond], 1).float(), t_b)
        a_t, a_prev, s_t = alphas[i], alphas_prev[i], sigmas[i]
        z0 = (z - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
        dir_term = torch.sqrt(1.0 - a_prev - s_t**2) * eps
        noise = s_t * torch.randn_like(z) if i>0 else 0.0
        z = torch.sqrt(a_prev) * z0 + dir_term + noise

    x_pred_m11 = vae_wrap.decode(z)
    return x_pred_m11


In [10]:
# =========================
# EMA
# =========================
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay=decay; self.shadow={}; self.backup={}
        for n,p in model.named_parameters():
            if p.requires_grad: self.shadow[n]=p.data.clone()
    def update(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n]=(1.0-self.decay)*p.data + self.decay*self.shadow[n]
    def apply_shadow(self, model):
        self.backup={}
        for n,p in model.named_parameters():
            if p.requires_grad:
                self.backup[n]=p.data.clone(); p.data=self.shadow[n]
    def restore(self, model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data=self.backup[n]
        self.backup={}


In [11]:

# =========================
# Save/Load
# =========================
def save_checkpoint(model, cond_enc, ema, opt, epoch, cfg, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "cond_enc": cond_enc.state_dict(),
        "ema": None if ema is None else ema.shadow,
        "opt": opt.state_dict(),
        "epoch": epoch,
        "cfg": cfg,
    }, path)

def load_checkpoint(model, cond_enc, ema, opt, path, device):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"], strict=True)
    cond_enc.load_state_dict(ckpt["cond_enc"], strict=True)
    if ema is not None and ckpt.get("ema", None) is not None:
        for n,p in model.named_parameters():
            if p.requires_grad and n in ckpt["ema"]:
                ema.shadow[n] = ckpt["ema"][n].clone()
    if opt is not None and "opt" in ckpt: opt.load_state_dict(ckpt["opt"])
    return ckpt.get("epoch", 0)



In [12]:
# =========================
# Main (Latent training)
# =========================
def main(cfg=CFG):
    set_seed(cfg["seed"])
    device = cfg["device"]
    os.makedirs(cfg["save_dir"], exist_ok=True)

    # Auto-resume only when available
    if not cfg.get("resume"):
        auto_resume = os.path.join(cfg["save_dir"], "last.pt")
        cfg["resume"] = auto_resume if os.path.exists(auto_resume) else None

    # Data
    ds = CTMRIFolderDataset(cfg)
    dl = DataLoader(ds, batch_size=cfg["batch_size"], shuffle=True,
                    num_workers=cfg["num_workers"], pin_memory=True, drop_last=True)

    # VAE (frozen)
    vae_wrap = build_vae(cfg).to(device)
    vae_wrap.eval()

    # 한 번 encode해서 latent 해상도 추정
    with torch.no_grad():
        tmp = next(iter(dl))
        mri_m11 = tmp["mri_m11"].to(device)
        z0 = vae_wrap.encode(mri_m11)
        lat_h, lat_w = z0.shape[-2], z0.shape[-1]

    # Latent U-Net & CT cond encoder
    net, cond_enc = build_latent_unet(cfg, latent_shape_hw=lat_h)
    net = net.to(device); cond_enc = cond_enc.to(device)
    sched = build_schedule(cfg["num_timesteps"], cfg["beta_schedule"], device)
    opt = optim.AdamW(list(net.parameters()) + list(cond_enc.parameters()), lr=cfg["lr"])
    ema = EMA(net, decay=cfg["ema_decay"])

    # Aux losses
    use_tv = cfg["aux_losses"]["use_tv"]
    tv_w   = cfg["aux_losses"]["tv_weight"]
    use_lp = cfg["aux_losses"]["use_lpips"] and HAS_LPIPS
    lpips  = LPIPS().to(device).eval() if use_lp else None

    start_epoch = 0
    if cfg["resume"] and os.path.exists(cfg["resume"]):
        start_epoch = load_checkpoint(net, cond_enc, ema, opt, cfg["resume"], device)
        print(f"[Resume] {cfg['resume']} @ epoch {start_epoch}")

    mse = nn.MSELoss()

    # ===== Train (FP32) =====
    net.train(); cond_enc.train()
    for epoch in range(start_epoch, cfg["epochs"]):
        t0 = time.time()
        for step, batch in enumerate(dl):
            ct_m11  = batch["ct_m11"].to(device)     # (B,1,H,W) [-1,1]
            mri_m11 = batch["mri_m11"].to(device)    # (B,1,H,W) [-1,1]

            # latent encode (no grad)
            with torch.no_grad():
                z0 = vae_wrap.encode(mri_m11)        # (B,Cz,h,w)

            # diffusion timestep & q(z_t | z0)
            t = torch.randint(0, cfg["num_timesteps"], (ct_m11.size(0),), device=device).long()
            z_t, noise = q_sample_latent(z0, t, sched)

            # cond
            ct_lat = cond_enc(ct_m11)                # (B,Cc,h,w)

            # predict eps in latent
            eps_pred = net(torch.cat([z_t, ct_lat], 1).float(), t)   # (B,Cz,h,w)

            # loss: latent ε-MSE
            loss = mse(eps_pred, noise)

            # optional: reconstruct x0_hat for aux losses
            if use_tv or use_lp:
                with torch.no_grad():
                    a_bar = sched.alphas_cumprod[t].view(-1,1,1,1)
                    z0_hat = (z_t - torch.sqrt(1 - a_bar) * eps_pred) / torch.sqrt(a_bar)
                    x0_hat_m11 = vae_wrap.decode(z0_hat)     # (B,1,H,W) [-1,1]

                if use_tv:
                    tv = (x0_hat_m11[:,:,1:,:]-x0_hat_m11[:,:,:-1,:]).abs().mean() + \
                         (x0_hat_m11[:,:,:,1:]-x0_hat_m11[:,:,:,:-1]).abs().mean()
                    loss = loss + tv_w * tv

                if use_lp and lpips is not None:
                    # LPIPS expects [-1,1] 3ch; 간단히 동일 채널 복제
                    x3 = x0_hat_m11.repeat(1,3,1,1); y3 = mri_m11.repeat(1,3,1,1)
                    loss = loss + cfg["aux_losses"]["lpips_weight"] * lpips(x3, y3).mean()

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            ema.update(net)

            if (step+1) % 50 == 0:
                print(f"[{epoch+1}/{cfg['epochs']}] step {step+1}/{len(dl)} loss {loss.item():.4f}")

        # ===== save + quick samples =====
        save_checkpoint(net, cond_enc, ema, opt, epoch+1, cfg, os.path.join(cfg["save_dir"], "last.pt"))

        ema.apply_shadow(net); net.eval(); cond_enc.eval()
        with torch.no_grad():
            # small sample set
            n_show = min(4, len(ds))
            sample_ct_m11 = torch.stack([ds[i]["ct_m11"] for i in range(n_show)], 0).to(device)
            if cfg["inference"]["sampler"].lower()=="ddpm":
                pred = sample_ddpm_latent(net, cond_enc, vae_wrap, sample_ct_m11, cfg["img_size"], sched, device=device)
                tag  = "ddpm"
            else:
                pred = sample_ddim_latent(net, cond_enc, vae_wrap, sample_ct_m11, cfg["img_size"], sched,
                                          ddim_steps=cfg["ddim_steps"], eta=cfg["ddim_eta"], device=device)
                tag  = f"ddim_eta{cfg['ddim_eta']:.1f}"
            for i in range(n_show):
                to_image_from_m11(pred[i]).save(os.path.join(cfg["save_dir"], f"ep{epoch+1:03d}_{tag}_{i}.png"))

        # epoch inference (Triptych with GT)
        # run_epoch_inference_ldm(net, cond_enc, vae_wrap, sched, cfg, ds, device, epoch+1)
        # ✅ 20에폭 단위 실행
        every = cfg["inference"].get("every_n_epochs", 1)
        if cfg["inference"].get("enabled", False) and ((epoch + 1) % every == 0):
            run_epoch_inference_ldm(net, cond_enc, vae_wrap, sched, cfg, ds, device, epoch+1)

        # restore & next
        net.train(); cond_enc.train(); ema.restore(net)
        print(f"Epoch {epoch+1} done in {time.time()-t0:.1f}s")

    print("Training finished.")


In [13]:
# =========================
# Epoch Inference (CT-only, Triptych)
# =========================
def _prepare_infer_cts(cfg, ds, device):
    inf = cfg["inference"]; imgs, stems, paths = [], [], []
    if inf["ct_list"]:
        paths = inf["ct_list"]
    elif inf["ct_glob"]:
        import glob
        # ✅ 전체 사용: 슬라이싱 제거
        paths = sorted(glob.glob(inf["ct_glob"]))
        # (선택) 너무 많으면 제한을 두고 싶을 때만 사용:
        # if inf.get("limit"):
        #     paths = paths[:inf["limit"]]
    else:
        n = min(inf["num_examples"], len(ds))
        paths = [ds.pairs[i][0] for i in range(n)]

    for p in paths:
        arr = imread_gray(p, cfg["img_size"])
        imgs.append(to_range_m11(to_tensor01(arr)))
        stems.append(os.path.splitext(os.path.basename(p))[0])
    return torch.stack(imgs, 0).to(cfg["device"]), stems, paths


@torch.no_grad()
def run_epoch_inference_ldm(net, cond_enc, vae_wrap, sched, cfg, ds, device, epoch_idx):
    inf = cfg["inference"]
    if not inf.get("enabled", False): return
    if inf["seed"] is not None:
        torch.manual_seed(inf["seed"]); torch.cuda.manual_seed_all(inf["seed"])

    ct_m11, stems, paths = _prepare_infer_cts(cfg, ds, device)
    H = cfg["img_size"]
    outdir = os.path.join(cfg["save_dir"], inf["out_subdir"]); os.makedirs(outdir, exist_ok=True)
    ct2gt = dict(ds.pairs)

    if inf["sampler"].lower()=="ddpm":
        pred_m11 = sample_ddpm_latent(net, cond_enc, vae_wrap, ct_m11, H, sched, device=device)
        tag="ddpm"
    else:
        pred_m11 = sample_ddim_latent(net, cond_enc, vae_wrap, ct_m11, H, sched,
                                      ddim_steps=inf["ddim_steps"], eta=inf["eta"], device=device)
        tag=f"ddim_eta{inf['eta']:.1f}"

    if inf.get("write_mapping", False):
        with open(os.path.join(outdir, f"ep{epoch_idx:03d}_mapping.txt"), "w") as f:
            for s,p in zip(stems, paths): f.write(f"{s} -> {p}\n")

    for i, s in enumerate(stems):
        pid = os.path.basename(os.path.dirname(os.path.dirname(paths[i])))  # eval/<ID>/ct_axial/<stem>.jpg
        base = f"ep{epoch_idx:03d}_{pid}_{s}_{tag}"
        to_image_from_m11(pred_m11[i]).save(os.path.join(outdir, f"{base}.png"))
    
        gt_path = ct2gt.get(paths[i], None)
        save_triptych(ct_m11[i], pred_m11[i], gt_path,
                      os.path.join(outdir, f"{base}_triptych.png"),
                      size=H)


In [16]:
# VAE 경로 지정
CFG["vae"]["config"] = "autoencoder_kl_64x64x3.yaml"
CFG["vae"]["ckpt"]   = "ae_model.ckpt"

CFG["inference"].update({
    "ct_list": [],
    # "ct_glob": "/data/ct-mri/dataset/processed_final_5/eval/*/ct_axial/*.jpg",
    "ct_glob": "../data/processed_with_skullstrip/eval/*/ct_axial/*.png",
    "write_mapping": True  # 매핑 파일 남기기
})
CFG["inference"]["every_n_epochs"] = 20

CFG["latent_unet"]["z_channels"] = 3                 # vq-f4의 embed_dim = 3

In [ ]:
main(CFG)

[Dataset] discovered pairs: 5910
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels


/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/youkyoung/.conda/envs/med/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/youkyoung/CT2MRI/diffusion_model/taming/modules/losses/lpips.py:29: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github

loaded pretrained LPIPS loss from taming/modules/autoencoder/lpips/vgg.pth
[1/50] step 50/738 loss 0.2352
[1/50] step 100/738 loss 0.3113
[1/50] step 150/738 loss 0.1018
[1/50] step 200/738 loss 0.2392
[1/50] step 250/738 loss 0.1052
[1/50] step 300/738 loss 0.1113
[1/50] step 350/738 loss 0.1213
[1/50] step 400/738 loss 0.0726
[1/50] step 450/738 loss 0.0941
[1/50] step 500/738 loss 0.0703
[1/50] step 550/738 loss 0.0556
[1/50] step 600/738 loss 0.0235
[1/50] step 650/738 loss 0.0819
[1/50] step 700/738 loss 0.0616
Epoch 1 done in 233.6s
[2/50] step 50/738 loss 0.1106
[2/50] step 100/738 loss 0.0409
[2/50] step 150/738 loss 0.1239
[2/50] step 200/738 loss 0.1084
[2/50] step 250/738 loss 0.0611


In [ ]:
# ==== Eval-only Inference (robust, no dependency on build_latent_unet) =======
import os, glob, torch
from PIL import Image

# ---------------- user settings ----------------
CT2MRI_CKPT = os.path.join(CFG["save_dir"], "last.pt")  # 학습 ckpt
# CT_GLOB     = "/data/ct-mri/dataset/processed_final_5/eval/*/ct_axial/*.jpg"
CT_GLOB     = "../data/processed_with_skullstrip/eval/*/ct_axial/*.png"
OUT_DIR     = os.path.join(CFG["save_dir"], "inference_eval")
SAMPLER     = "ddim"      # "ddim" or "ddpm"
DDIM_STEPS  = 200
ETA         = 0.0
BATCH       = 4
DEVICE      = CFG["device"]
# ------------------------------------------------

# 헬퍼: [-1,1] 변환 (없으면 정의)
if "to_range_m11" in globals():
    to_m11 = to_range_m11
else:
    def to_m11(x01: torch.Tensor) -> torch.Tensor:
        return x01 * 2.0 - 1.0

# 필요한 import (의존 최소화)
from ldm.modules.diffusionmodules.openaimodel import UNetModel
try:
    from ldm.modules.diffusionmodules.util import (
        make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters
    )
except Exception:
    from ldm.modules.diffusionmodules.utils import (
        make_beta_schedule, make_ddim_timesteps, make_ddim_sampling_parameters
    )

# 스케줄 & 샘플러 (latent)
def build_schedule(num_steps: int, beta_schedule="cosine", device="cpu"):
    betas = torch.from_numpy(make_beta_schedule(beta_schedule, num_steps)).float().to(device)
    alphas = 1.0 - betas
    a_bar = torch.cumprod(alphas, dim=0)
    a_bar_prev = torch.cat([torch.tensor([1.0], device=device), a_bar[:-1]], dim=0)
    return {
        "betas": betas, "alphas": alphas,
        "a_bar": a_bar, "a_bar_prev": a_bar_prev,
        "sqrt_a_bar": torch.sqrt(a_bar),
        "sqrt_one_minus_a_bar": torch.sqrt(1.0 - a_bar),
        "post_var": betas * (1.0 - a_bar_prev) / (1.0 - a_bar)
    }

@torch.no_grad()
def sample_ddpm_latent(model, cond_enc, vae, ct_m11, sched, device="cpu"):
    B = ct_m11.size(0)
    ct_lat = cond_enc(ct_m11)
    Cz, h, w = model.out_channels, ct_lat.shape[-2], ct_lat.shape[-1]
    z = torch.randn(B, Cz, h, w, device=device)
    T = sched["betas"].size(0)
    for t in reversed(range(T)):
        t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([z, ct_lat],1).float(), t_b)
        a_bar_t = sched["a_bar"][t]
        z0 = (z - torch.sqrt(1 - a_bar_t) * eps) / torch.sqrt(a_bar_t)
        mean = torch.sqrt(sched["a_bar_prev"][t]) * z0 + torch.sqrt(1.0 - sched["a_bar_prev"][t]) * eps
        z = mean + (torch.sqrt(sched["post_var"][t]) * torch.randn_like(z) if t>0 else 0)
    return vae.decode(z)

@torch.no_grad()
def sample_ddim_latent(model, cond_enc, vae, ct_m11, sched, ddim_steps=200, eta=0.0, device="cpu"):
    B = ct_m11.size(0)
    ct_lat = cond_enc(ct_m11)
    Cz, h, w = model.out_channels, ct_lat.shape[-2], ct_lat.shape[-1]
    z = torch.randn(B, Cz, h, w, device=device)
    ts = torch.tensor(make_ddim_timesteps('uniform', ddim_steps, sched["betas"].size(0))) - 1
    sigmas, alphas, alphas_prev = make_ddim_sampling_parameters(sched["a_bar"].cpu().numpy(), ts.numpy(), eta)
    sigmas = torch.from_numpy(sigmas).float().to(device)
    alphas = torch.from_numpy(alphas).float().to(device)
    alphas_prev = torch.from_numpy(alphas_prev).float().to(device)
    for i in reversed(range(len(ts))):
        t = int(ts[i]); t_b = torch.full((B,), t, device=device, dtype=torch.long)
        eps = model(torch.cat([z, ct_lat],1).float(), t_b)
        a_t, a_prev, s_t = alphas[i], alphas_prev[i], sigmas[i]
        z0 = (z - torch.sqrt(1.0 - a_t) * eps) / torch.sqrt(a_t)
        dir_term = torch.sqrt(1.0 - a_prev - s_t**2) * eps
        noise = s_t * torch.randn_like(z) if i>0 else 0.0
        z = torch.sqrt(a_prev) * z0 + dir_term + noise
    return vae.decode(z)

# Triptych 저장 (이미 위에 있으면 사용)
def to_img_from_m11(x: torch.Tensor) -> Image.Image:
    x = x.detach().clamp(-1,1); x = (x+1.0)*0.5
    return Image.fromarray((x[0].cpu().numpy()*255).round().astype(np.uint8), "L")

def save_triptych(ct_m11: torch.Tensor, pred_m11: torch.Tensor, gt_path, out_path: str, size: int):
    ct_p   = to_img_from_m11(ct_m11)
    pred_p = to_img_from_m11(pred_m11)
    if gt_path and os.path.exists(gt_path):
        arr = imread_gray(gt_path, size)
        gt_p = Image.fromarray((arr*255).astype(np.uint8), "L")
    else:
        gt_p = Image.new("L", (size,size), 0)
    panel = Image.new("L", (size*3, size))
    panel.paste(ct_p,(0,0)); panel.paste(gt_p,(size,0)); panel.paste(pred_p,(size*2,0))
    os.makedirs(os.path.dirname(out_path), exist_ok=True); panel.save(out_path)

# 1) ckpt & cfg
ckpt = torch.load(CT2MRI_CKPT, map_location="cpu")
cfg  = ckpt["cfg"]; img_size = cfg["img_size"]

# 2) VAE 준비 (위 셀의 build_vae(CFG) 사용)
vae_wrap = build_vae(CFG).to(DEVICE).eval()

# 3) eval CT 리스트
ct_paths_all = sorted(glob.glob(CT_GLOB))
assert len(ct_paths_all)>0, f"No CT matched: {CT_GLOB}"

# 4) 잠재 해상도/채널
x01 = to_tensor01(imread_gray(ct_paths_all[0], img_size)).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    z0 = vae_wrap.encode(to_m11(x01))
z_channels = z0.shape[1]; lat_h, lat_w = z0.shape[-2], z0.shape[-1]

# 5) U-Net + CondEncoder 직접 생성 (호환성 모드)
lu = cfg["latent_unet"]
in_ch  = z_channels + lu["cond_channels"]
out_ch = z_channels
net = UNetModel(
    image_size=lat_h, in_channels=in_ch, out_channels=out_ch,
    model_channels=lu["model_channels"], num_res_blocks=lu["num_res_blocks"],
    channel_mult=tuple(lu["channel_mult"]),
    attention_resolutions=tuple(lu.get("attention_resolutions", (16,))),
    num_heads=lu.get("num_heads",-1), num_head_channels=lu.get("num_head_channels",64),
    num_heads_upsample=lu.get("num_heads_upsample",-1), dropout=lu["dropout"],
    conv_resample=True, dims=2, num_classes=None,
    use_checkpoint=lu["use_checkpoint"], use_fp16=lu["use_fp16"],
    use_scale_shift_norm=lu["use_scale_shift_norm"], resblock_updown=lu["resblock_updown"],
    use_new_attention_order=lu["use_new_attention_order"],
    use_spatial_transformer=lu["use_spatial_transformer"],
    transformer_depth=lu["transformer_depth"], context_dim=lu["context_dim"],
    n_embed=lu["n_embed"], legacy=lu["legacy"]
).to(DEVICE).eval()

# Cond encoder (간단 conv; 학습 때와 동일 규격)
class CTCondEncoder(torch.nn.Module):
    def __init__(self, cond_channels=1):
        super().__init__()
        self.enc = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, 3, 2, 1), torch.nn.SiLU(),
            torch.nn.Conv2d(32, 64, 3, 2, 1), torch.nn.SiLU(),
            torch.nn.Conv2d(64, cond_channels, 1)
        )
    def forward(self, x): return self.enc(x)

cond_enc = CTCondEncoder(cond_channels=lu["cond_channels"]).to(DEVICE).eval()

# 6) 가중치 로드 + EMA 적용
net.load_state_dict(ckpt["model"], strict=True)
cond_enc.load_state_dict(ckpt["cond_enc"], strict=True)
if ckpt.get("ema", None) is not None:
    with torch.no_grad():
        for n,p in net.named_parameters():
            if p.requires_grad and n in ckpt["ema"]:
                p.data.copy_(ckpt["ema"][n].to(p.device))

# 7) 스케줄
sched = build_schedule(cfg["num_timesteps"], cfg["beta_schedule"], DEVICE)

# 8) 인퍼런스
os.makedirs(OUT_DIR, exist_ok=True)
with open(os.path.join(OUT_DIR, "mapping.txt"), "w") as mapfile:
    print(f"[INFO] matched CT files: {len(ct_paths_all)} | saving to: {OUT_DIR}")
    for i in range(0, len(ct_paths_all), BATCH):
        batch_paths = ct_paths_all[i:i+BATCH]
        imgs_m11, stems, pids = [], [], []
        for p in batch_paths:
            arr = imread_gray(p, img_size)
            imgs_m11.append(to_m11(to_tensor01(arr)))
            stems.append(os.path.splitext(os.path.basename(p))[0])
            pids.append(os.path.basename(os.path.dirname(os.path.dirname(p))))
        ct_m11 = torch.stack(imgs_m11, 0).to(DEVICE)

        if SAMPLER.lower()=="ddpm":
            pred_m11 = sample_ddpm_latent(net, cond_enc, vae_wrap, ct_m11, sched, device=DEVICE)
            tag="ddpm"
        else:
            pred_m11 = sample_ddim_latent(net, cond_enc, vae_wrap, ct_m11, sched,
                                          ddim_steps=DDIM_STEPS, eta=ETA, device=DEVICE)
            tag=f"ddim_eta{ETA:.1f}"

        for k, stem in enumerate(stems):
            base = f"{pids[k]}_{stem}_{tag}"
            to_img_from_m11(pred_m11[k]).save(os.path.join(OUT_DIR, f"{base}.png"))
            gt_path = os.path.join(os.path.dirname(os.path.dirname(batch_paths[k])),
                                   "mri_b0_axial", f"{stem}.png")
            save_triptych(ct_m11[k], pred_m11[k], gt_path if os.path.exists(gt_path) else None,
                          os.path.join(OUT_DIR, f"{base}_triptych.png"), size=img_size)
            mapfile.write(f"{base} <- {batch_paths[k]}\n")

print("[Done]")
